# Figure 2 — public-grid and self-reliant electricity risk

Draft extracted from `moz_paper_figure.ipynb`, primarily cell index 14.
It intentionally uses a real 2 × 2 layout rather than the legacy 2 × 4
canvas that was later cropped for the manuscript.

The loader below uses the current 2,703-track public-grid summary and the
independent-solar track outputs directly. It therefore does not require the
old hand-prepared GeoPackages. Paths can be overridden with the same
`INFRACASCADE_*` variables used by the Slurm workflow.

The annualization factor, labels, and final typography remain
manuscript-editing points.


In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("INFRACASCADE_PROJECT_DIR", Path.cwd())).resolve()
if PROJECT_ROOT.name == "moz_notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = Path(os.environ.get("MOZ_FIGURE_DATA_ROOT", PROJECT_ROOT / "figure_data"))
FIGURE_DIR = Path(os.environ.get(
    "MOZ_MANUSCRIPT_FIGURE_DIR",
    PROJECT_ROOT / "outputs" / "manuscript_figures",
))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Data:", DATA_ROOT)
print("Figures:", FIGURE_DIR)


In [ ]:
from capacity_ablation_figure import (
    attach_hospital_geometry,
    load_boundary,
    load_hospital_data,
    load_power_summary,
    load_solar_summary,
    validate_standard_cohorts,
)
from synthetic_tracks import expected_track_columns

SCRATCH_ROOT = Path(os.environ.get("SCRATCH", "/scratch/users/zywei"))
WIND_ROOT = Path(os.environ.get(
    "INFRACASCADE_WIND_ROOT", SCRATCH_ROOT / "wind_result"
))
HOSPITAL_PATH = Path(os.environ.get(
    "INFRACASCADE_HOSPITAL_DATA", PROJECT_ROOT / "SARA_processed"
))
PUBLIC_SUMMARY_PATH = Path(os.environ.get(
    "INFRACASCADE_PUBLIC_SUMMARY", WIND_ROOT / "df_final_public_25.csv"
))
SOLAR_ROOT = Path(os.environ.get(
    "INFRACASCADE_SOLAR_ROOT", WIND_ROOT / "synthetic_solar"
))
ALL_GRID_SUMMARY_PATH = Path(os.environ.get(
    "INFRACASCADE_ALL_GRID_SUMMARY", WIND_ROOT / "df_final_all_grid_25.csv"
))
WORLD_PATH = Path(os.environ.get(
    "INFRACASCADE_BOUNDARY_DATA", SCRATCH_ROOT / "World_Countries"
))

hospital = load_hospital_data(HOSPITAL_PATH)
public_summary = load_power_summary(
    PUBLIC_SUMMARY_PATH, require_complete=True, allow_unavailable=True
)
solar_summary = load_solar_summary(SOLAR_ROOT, hospital, zero_empty_events=True)
# This check also confirms that the public and solar cohorts form the same
# 1,380-hospital universe used by the all-grid ablation.
all_grid_summary = load_power_summary(ALL_GRID_SUMMARY_PATH, require_complete=True)
validate_standard_cohorts(public_summary, all_grid_summary, solar_summary)

gdf_elec = attach_hospital_geometry(public_summary, hospital)
solar = attach_hospital_geometry(solar_summary, hospital)
mozambique = load_boundary(WORLD_PATH)

def facility_key(value):
    text = str(value).strip()
    return text[:-2] if text.endswith(".0") else text

urban = pd.to_numeric(hospital["urban"], errors="coerce")
urban_ids = set(hospital.loc[urban.eq(1), "facility_i"].map(facility_key))
rural_ids = set(hospital.loc[urban.eq(2), "facility_i"].map(facility_key))
generator_ids = set(
    hospital.loc[
        hospital["has_genera"].astype(str).str.strip().str.lower().eq("yes"),
        "facility_i",
    ].map(facility_key)
)
gdf_elec["facility_key"] = gdf_elec["facility_i"].map(facility_key)
solar["facility_key"] = solar["facility_i"].map(facility_key)

gdf_elec = gdf_elec.copy()
solar = solar.copy()
track_columns = list(expected_track_columns())
ANNUALIZATION_FACTOR = len(track_columns) / 10_000
gdf_elec["mean"] = gdf_elec[track_columns].where(
    gdf_elec[track_columns] >= 0
).mean(axis=1).fillna(-1.0)
solar["mean"] = solar[track_columns].mean(axis=1)
gdf_elec["annual_probability"] = gdf_elec["mean"] * ANNUALIZATION_FACTOR
solar["annual_probability"] = solar["mean"] * ANNUALIZATION_FACTOR


In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

cmap = "BuGn"
norm = Normalize(0, 0.05)
fig, axes = plt.subplots(2, 2, figsize=(6.5, 8), sharex=True, sharey=True)

for axis in axes.flat:
    mozambique.plot(ax=axis, color="gainsboro", edgecolor="black", linewidth=0.8)
    axis.set(xlim=(30, 41), ylim=(-27, -10))
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_aspect("equal")

panels = [
    (axes[0, 0], gdf_elec, urban_ids, "(a)", "Urban"),
    (axes[0, 1], gdf_elec, rural_ids, "(b)", "Rural"),
    (axes[1, 0], solar, urban_ids, "(c)", ""),
    (axes[1, 1], solar, rural_ids, "(d)", ""),
]
for axis, layer, facility_ids, panel, title in panels:
    selected = layer[
        layer["facility_key"].isin(facility_ids) & layer["mean"].ge(0)
    ]
    selected.plot(
        ax=axis,
        column="annual_probability",
        cmap=cmap,
        norm=norm,
        markersize=15,
        linewidth=0,
    )
    with_generator = selected[selected["facility_key"].isin(generator_ids)]
    if not with_generator.empty:
        with_generator.plot(
            ax=axis,
            facecolor="none",
            edgecolor="deeppink",
            markersize=18,
            linewidth=0.5,
        )
    axis.text(0.02, 0.98, panel, transform=axis.transAxes, va="top")
    axis.set_title(title)

fig.text(0.015, 0.70, "Depend on public infrastructure", rotation=90, va="center")
fig.text(0.015, 0.29, "Self-reliant", rotation=90, va="center")

colorbar = fig.colorbar(
    cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, fraction=0.035, pad=0.025
)
colorbar.set_label("Annualized electricity disruption index (yr$^{-1}$)")
colorbar.set_ticks(np.arange(0, 0.051, 0.01), labels=[f"{x:.2f}" for x in np.arange(0, 0.051, 0.01)])

fig.legend(
    handles=[Line2D([], [], marker="o", linestyle="none", markerfacecolor="none",
                    markeredgecolor="deeppink", label="Has generator")],
    loc="lower center",
    frameon=False,
)
fig.subplots_adjust(left=0.10, right=0.88, bottom=0.08, top=0.94, wspace=0.03, hspace=0.03)

for suffix in ("pdf", "png"):
    fig.savefig(FIGURE_DIR / f"figure_02.{suffix}", dpi=300, bbox_inches="tight")
plt.show()
